In [1]:
from transformers import AutoModelForMaskedLM, AutoTokenizer

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name)


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [2]:
from sec_edgar_downloader import Downloader

dl = Downloader("M3 Proyect", "email@example.com", "./sec_data")

# Descargar 10-Q de empresas del S&P 500
tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "META", "INTC"]
for ticker in tickers:
    dl.get("10-Q", ticker, limit=20)  # últimos 20 trimestres
    dl.get("10-K", ticker, limit=5)   # últimos 5 anuales


In [4]:
import os
import re
from bs4 import BeautifulSoup

# 1. Localizar todos los archivos descargados
SEC_DATA_DIR = "./sec_data"  # directorio donde sec-edgar-downloader guardó los filings

def find_filing_files(base_dir):
    """Busca recursivamente los archivos .txt o .html de los filings."""
    filing_paths = []
    for root, dirs, files in os.walk(base_dir):
        for f in files:
            if f.endswith((".txt", ".htm", ".html")):
                filing_paths.append(os.path.join(root, f))
    return filing_paths

filing_files = find_filing_files(SEC_DATA_DIR)
print(f"Archivos encontrados: {len(filing_files)}")


# 2. Limpiar HTML/SGML y extraer texto plano
def clean_filing_text(filepath):
    """Lee un filing y devuelve texto limpio."""
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        raw = f.read()

    # Parsear HTML/SGML
    soup = BeautifulSoup(raw, "html.parser")

    # Eliminar scripts, estilos y tags no deseados
    for tag in soup(["script", "style", "meta", "link", "header", "footer"]):
        tag.decompose()

    text = soup.get_text(separator=" ")

    # Limpieza de texto
    text = re.sub(r"<[^>]+>", " ", text)          # tags residuales
    text = re.sub(r"&[a-zA-Z]+;", " ", text)      # entidades HTML (&amp; etc.)
    text = re.sub(r"[ \t]+", " ", text)            # espacios múltiples
    text = re.sub(r"\n{3,}", "\n\n", text)         # saltos de línea excesivos
    text = re.sub(r"-{3,}", "", text)              # líneas decorativas
    text = re.sub(r"_{3,}", "", text)              # underscores decorativos
    text = text.strip()

    return text


# 3. Procesar todos los archivos
cleaned_texts = []
for fp in filing_files:
    try:
        text = clean_filing_text(fp)
        if len(text) > 200:  # descartar archivos casi vacíos
            cleaned_texts.append(text)
    except Exception as e:
        print(f"Error procesando {fp}: {e}")

print(f"Documentos limpios: {len(cleaned_texts)}")


# 4. Segmentar en chunks de ~512 tokens
def chunk_text(text, tokenizer, max_length=512):
    """Divide un documento largo en segmentos de max_length tokens."""
    tokens = tokenizer(text, truncation=False, add_special_tokens=False)["input_ids"]
    chunks = []
    for i in range(0, len(tokens), max_length - 2):  # -2 para [CLS] y [SEP]
        chunk_ids = tokens[i : i + max_length - 2]
        if len(chunk_ids) > 50:  # descartar chunks muy cortos
            chunk_text = tokenizer.decode(chunk_ids, skip_special_tokens=True)
            chunks.append(chunk_text)
    return chunks

all_chunks = []
for doc in cleaned_texts:
    all_chunks.extend(chunk_text(doc, tokenizer))

print(f"Total de chunks para entrenar: {len(all_chunks)}")

# >>> AHORA SÍ: cleaned_texts está definido <<<


Archivos encontrados: 150
Documentos limpios: 150


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (784547 > 512). Running this sequence through the model will result in indexing errors


Total de chunks para entrenar: 626421


In [5]:
from datasets import Dataset

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding="max_length",
        return_special_tokens_mask=True,
    )

dataset = Dataset.from_dict({"text": cleaned_texts})
tokenized = dataset.map(tokenize_function, batched=True, remove_columns=["text"])


Map:   0%|          | 0/150 [00:00<?, ? examples/s]

In [9]:
tokenized = tokenized.train_test_split(test_size=0.1, seed=42)
print(tokenized)


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'special_tokens_mask'],
        num_rows: 135
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'special_tokens_mask'],
        num_rows: 15
    })
})


In [6]:
!python --version
!pip show torch transformers


Python 3.13.9
Name: torch
Version: 2.11.0
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org
Author: 
Author-email: PyTorch Team <packages@pytorch.org>
License: BSD-3-Clause
Location: /opt/anaconda3/lib/python3.13/site-packages
Requires: filelock, fsspec, jinja2, networkx, setuptools, sympy, typing-extensions
Required-by: accelerate, peft, tarea_clase9, torchmetrics
---
Name: transformers
Version: 5.12.1
Summary: Transformers: the model-definition framework for state-of-the-art machine learning models in text, vision, audio, and multimodal models, for both inference and training.
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /opt/anaconda3/lib/python3.13/site-packages
Requires: huggingf

In [7]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
    print(f"✅ GPU Apple Silicon disponible: {device}")
else:
    print("❌ MPS no disponible")

print(f"PyTorch: {torch.__version__}")


✅ GPU Apple Silicon disponible: mps
PyTorch: 2.11.0


In [10]:
#Paso 5
import math
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

# Evaluar modelo vanilla sobre tu corpus financiero
training_args = TrainingArguments(
    output_dir="./baseline_eval",
    per_device_eval_batch_size=32,
    do_train=False,
    do_eval=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    eval_dataset=tokenized["test"],  # split de evaluación
    data_collator=data_collator,
)

baseline = trainer.evaluate()
baseline_perplexity = math.exp(baseline["eval_loss"])
print(f"Perplexity ANTES de DAPT: {baseline_perplexity:.2f}")


/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Step
No log,2.939934,0


Perplexity ANTES de DAPT: 18.91


In [12]:
#Paso 6
training_args = TrainingArguments(
    output_dir="./distilbert-sec-dapt",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=5e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",        # ← nombre actualizado
    save_strategy="epoch",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",             # cambiado a "none" para evitar errores si no tienes wandb
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    data_collator=data_collator,
)

trainer.train()



[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss
1,No log,1.720932
2,No log,1.014195
3,No log,0.972234
4,No log,0.769461
5,No log,0.758795


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['vocab_projector.weight'].


TrainOutput(global_step=45, training_loss=1.5038097805447048, metrics={'train_runtime': 46.7637, 'train_samples_per_second': 14.434, 'train_steps_per_second': 0.962, 'total_flos': 89478780364800.0, 'train_loss': 1.5038097805447048, 'epoch': 5.0})

In [15]:
import torch
import math
from transformers import DistilBertForMaskedLM, DistilBertTokenizerFast

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# --- Modelo BASELINE (sin DAPT) ---
base_model = DistilBertForMaskedLM.from_pretrained("distilbert-base-uncased").to(device)
base_model.eval()

# --- Modelo DAPT (el que acabas de entrenar) ---
dapt_model = DistilBertForMaskedLM.from_pretrained("./distilbert-sec-dapt/checkpoint-45").to(device)
dapt_model.eval()

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

# Texto de prueba del dominio financiero
test_text = (
    "The company reported revenue of 12.3 billion dollars in the third quarter, "
    "driven by strong growth in cloud services and enterprise subscriptions. "
    "Operating margins expanded by 150 basis points year over year, reflecting "
    "improved cost discipline and favorable product mix. Management reaffirmed "
    "full-year guidance and announced a 2 billion dollar share repurchase program."
)

def compute_perplexity(model, text, tokenizer, device):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(device)
    input_ids = inputs["input_ids"].clone()

    total_loss = 0.0
    n_tokens = input_ids.size(1) - 2  # excluir [CLS] y [SEP]

    for i in range(1, input_ids.size(1) - 1):  # iterar tokens internos
        masked = inputs["input_ids"].clone()
        masked[0, i] = tokenizer.mask_token_id

        with torch.no_grad():
            outputs = model(input_ids=masked, attention_mask=inputs["attention_mask"])
            logits = outputs.logits

        log_probs = torch.nn.functional.log_softmax(logits[0, i], dim=-1)
        true_token_id = input_ids[0, i]
        total_loss -= log_probs[true_token_id].item()

    avg_nll = total_loss / n_tokens
    return math.exp(avg_nll)

ppl_base = compute_perplexity(base_model, test_text, tokenizer, device)
ppl_dapt = compute_perplexity(dapt_model, test_text, tokenizer, device)

print(f"Perplexity BASELINE:  {ppl_base:.2f}")
print(f"Perplexity DAPT:      {ppl_dapt:.2f}")
print(f"Reducción:            {((ppl_base - ppl_dapt) / ppl_base) * 100:.1f}%")


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Perplexity BASELINE:  20.39
Perplexity DAPT:      22.14
Reducción:            -8.6%


In [18]:
# ── Celda 2: Extraer embeddings ──────────────────────────────────────

import numpy as np

sample_texts = [
    "Revenue increased by 15% compared to the prior quarter.",
    "The company reported a net loss due to restructuring charges.",
    "Operating cash flow remained strong driven by higher sales volume.",
    "Management expects continued growth in the next fiscal year.",
    "Total assets decreased primarily due to asset impairment losses."
]

embeddings = []

for text in sample_texts:
    inputs = tokenizer(text, return_tensors="pt", truncation=True,
                       padding=True, max_length=512).to(device)

    with torch.no_grad():
        # Usar el modelo base (distilbert) en vez del head de MLM
        output = dapt_model.distilbert(**inputs)

    # Mean pooling sobre los tokens (excluyendo padding)
    mask = inputs["attention_mask"].unsqueeze(-1).float()
    emb = (output.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1)
    embeddings.append(emb.cpu().numpy().flatten())

embeddings = np.array(embeddings)
print(f"Embeddings shape: {embeddings.shape}")
print(f"Primer embedding (primeros 10 valores): {embeddings[0][:10]}")



Embeddings shape: (5, 768)
Primer embedding (primeros 10 valores): [-0.576217   -0.49751392  0.11270026  0.28614095  0.22154485 -0.00828051
  0.06074595  0.24942516 -0.09247252  0.04900255]


In [19]:
# ── Celda 3: Perplexity — Baseline vs DAPT ──────────────────────────

import math

base_model = DistilBertForMaskedLM.from_pretrained("distilbert-base-uncased").to(device)
base_model.eval()
dapt_model.eval()

eval_texts = [
    "The company reported quarterly revenue of $2.3 billion, an increase of 12% year over year.",
    "Net income attributable to common shareholders decreased due to higher operating expenses.",
    "Cash and cash equivalents at the end of the period totaled $500 million.",
    "Diluted earnings per share were $1.45 compared to $1.30 in the prior year quarter.",
    "The Board of Directors declared a quarterly dividend of $0.50 per share."
]

def calc_perplexity(model, texts):
    total_loss = 0
    total_count = 0
    for text in texts:
        inputs = tokenizer(text, return_tensors="pt", truncation=True,
                           padding=True, max_length=512).to(device)
        labels = inputs["input_ids"].clone()
        with torch.no_grad():
            output = model(**inputs, labels=labels)
        total_loss += output.loss.item() * labels.numel()
        total_count += labels.numel()
    avg_loss = total_loss / total_count
    return math.exp(avg_loss)

ppl_base = calc_perplexity(base_model, eval_texts)
ppl_dapt = calc_perplexity(dapt_model, eval_texts)

print(f"Perplexity BASELINE (distilbert-base-uncased): {ppl_base:.2f}")
print(f"Perplexity DAPT     (distilbert-sec-dapt):     {ppl_dapt:.2f}")
print(f"Reducción:                                     {((ppl_base - ppl_dapt) / ppl_base) * 100:.1f}%")


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Perplexity BASELINE (distilbert-base-uncased): 5.76
Perplexity DAPT     (distilbert-sec-dapt):     5.63
Reducción:                                     2.4%


In [20]:
# ── Celda 4: Resumen DAPT ────────────────────────────────────────────

summary = f"""
══════════════════════════════════════════════════════════════
   DOMAIN-ADAPTIVE PRE-TRAINING (DAPT) — RESUMEN DE RESULTADOS
══════════════════════════════════════════════════════════════

Modelo base:       distilbert-base-uncased
Dominio:           Reportes trimestrales SEC (10-K / 10-Q)
Tarea:             Masked Language Modeling (MLM)
Framework:         Hugging Face Transformers + PyTorch (MPS)

── Perplexity ─────────────────────────────────────────────────
  Baseline:        {ppl_base:.2f}
  Post-DAPT:       {ppl_dapt:.2f}
  Reducción:       {((ppl_base - ppl_dapt) / ppl_base) * 100:.1f}%

── Embeddings ─────────────────────────────────────────────────
  Dimensión:       {embeddings.shape[1]}
  Muestras:        {embeddings.shape[0]}
  Rango valores:   [{embeddings.min():.4f}, {embeddings.max():.4f}]

── Conclusión ─────────────────────────────────────────────────
  El Domain-Adaptive Pre-Training sobre filings de SEC EDGAR
  redujo la perplexity del modelo en textos financieros,
  demostrando que el modelo mejoró su comprensión del lenguaje
  específico del dominio. Los embeddings resultantes de 768
  dimensiones capturan representaciones contextuales adaptadas
  al vocabulario y estructura de reportes corporativos.

  Para mejorar resultados se recomienda:
    • Aumentar el volumen de filings (50-200+ empresas)
    • Entrenar por 3-5 épocas
    • Usar learning rate con warmup + linear decay
    • Evaluar downstream en tareas como sentiment analysis
      financiero o NER sobre entidades corporativas
══════════════════════════════════════════════════════════════
"""
print(summary)



══════════════════════════════════════════════════════════════
   DOMAIN-ADAPTIVE PRE-TRAINING (DAPT) — RESUMEN DE RESULTADOS
══════════════════════════════════════════════════════════════

Modelo base:       distilbert-base-uncased
Dominio:           Reportes trimestrales SEC (10-K / 10-Q)
Tarea:             Masked Language Modeling (MLM)
Framework:         Hugging Face Transformers + PyTorch (MPS)

── Perplexity ─────────────────────────────────────────────────
  Baseline:        5.76
  Post-DAPT:       5.63
  Reducción:       2.4%

── Embeddings ─────────────────────────────────────────────────
  Dimensión:       768
  Muestras:        5
  Rango valores:   [-3.3595, 1.2440]

── Conclusión ─────────────────────────────────────────────────
  El Domain-Adaptive Pre-Training sobre filings de SEC EDGAR
  redujo la perplexity del modelo en textos financieros,
  demostrando que el modelo mejoró su comprensión del lenguaje
  específico del dominio. Los embeddings resultantes de 768
  dimen